# Chapter 12 — Data Structures and Algorithms

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 12.

---


### Item 100 — Sort by Complex Criteria Using the `key` Parameter

`list.sort(key=func)` sorts by whatever `func(item)` returns, instead of comparing items directly — this handles sorting objects by an attribute, sorting by multiple criteria (return a tuple: `key=lambda p: (p.category, -p.price)`), or any custom order, without needing to define `__lt__` on the objects themselves.


In [1]:
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    category: str
    price: float

products = [
    Product('Widget', 'tools', 9.99),
    Product('Gadget', 'electronics', 49.99),
    Product('Gizmo', 'electronics', 19.99),
    Product('Hammer', 'tools', 14.99),
]

# Sort by a single attribute
by_price = sorted(products, key=lambda p: p.price)
print([p.name for p in by_price])

# Sort by MULTIPLE criteria: category first, then price descending within each category
by_category_then_price = sorted(products, key=lambda p: (p.category, -p.price))
print([(p.category, p.name, p.price) for p in by_category_then_price])


['Widget', 'Hammer', 'Gizmo', 'Gadget']
[('electronics', 'Gadget', 49.99), ('electronics', 'Gizmo', 19.99), ('tools', 'Hammer', 14.99), ('tools', 'Widget', 9.99)]


### Item 101 — Know the Difference Between `sort` and `sorted`

`list.sort()` sorts a list **in place** and returns `None` (writing `mylist = mylist.sort()` is a classic bug — it discards the list!); `sorted(iterable)` returns a **new** list and leaves the original (which doesn't even need to be a list — any iterable works) untouched. Use `.sort()` when you specifically want to mutate an existing list; use `sorted()` everywhere else, especially when you need to keep the original order too.


In [2]:
numbers = [3, 1, 4, 1, 5, 9, 2, 6]

# sorted(): returns a NEW list, original untouched
new_list = sorted(numbers)
print('original untouched:', numbers)
print('new sorted list:', new_list)

# .sort(): mutates IN PLACE, returns None
result = numbers.sort()
print('numbers is now sorted in place:', numbers)
print('the return value of .sort() is:', result)   # None -- classic bug if you do `x = x.sort()`

# sorted() also works on any iterable, not just lists
print(sorted('unsorted string'))
print(sorted({3, 1, 2}))


original untouched: [3, 1, 4, 1, 5, 9, 2, 6]
new sorted list: [1, 1, 2, 3, 4, 5, 6, 9]
numbers is now sorted in place: [1, 1, 2, 3, 4, 5, 6, 9]
the return value of .sort() is: None
[' ', 'd', 'e', 'g', 'i', 'n', 'n', 'o', 'r', 'r', 's', 's', 't', 't', 'u']
[1, 2, 3]


### Item 102 — Consider Searching Sorted Sequences with `bisect`

Searching for a value (or an insertion point) in an already-sorted list with a linear scan is O(n); the standard library's `bisect` module does it in O(log n) via binary search. `bisect.bisect_left(sorted_list, value)` finds where `value` would be inserted to keep the list sorted; `bisect.insort(sorted_list, value)` inserts it there directly.


In [3]:
import bisect

sorted_scores = [10, 25, 40, 55, 70, 85, 100]

# Binary search for an insertion point -- O(log n), not a linear scan
position = bisect.bisect_left(sorted_scores, 60)
print(f'60 would be inserted at index {position}, keeping the list sorted')

# insort inserts directly, maintaining sorted order automatically
bisect.insort(sorted_scores, 60)
print(sorted_scores)

# Practical use: grading via bucket boundaries
def grade(score, breakpoints=[60, 70, 80, 90], grades='FDCBA'):
    i = bisect.bisect(breakpoints, score)
    return grades[i]

for s in [55, 65, 75, 85, 95]:
    print(s, '->', grade(s))


60 would be inserted at index 4, keeping the list sorted
[10, 25, 40, 55, 60, 70, 85, 100]
55 -> F
65 -> D
75 -> C
85 -> B
95 -> A


### Item 103 — Prefer `deque` for Producer-Consumer Queues

A plain `list` used as a queue (`.pop(0)` to remove from the front) is O(n) per removal, since every remaining item has to shift down one position — this gets slow fast for a large queue. `collections.deque` supports O(1) appends/pops from **both** ends (`.append()`/`.appendleft()`, `.pop()`/`.popleft()`), making it the correct structure for FIFO queue behavior.


In [4]:
from collections import deque
import time

# list as a queue: .pop(0) is O(n) -- gets slower as the list grows
q_list = list(range(5000))
start = time.time()
while q_list:
    q_list.pop(0)
list_time = time.time() - start

# deque: .popleft() is O(1) regardless of size
q_deque = deque(range(5000))
start = time.time()
while q_deque:
    q_deque.popleft()
deque_time = time.time() - start

print(f'list.pop(0) x 5000:      {list_time:.4f}s')
print(f'deque.popleft() x 5000:  {deque_time:.4f}s (should be noticeably faster)')

dq = deque()
dq.append('right side')
dq.appendleft('left side')
print(dq)


list.pop(0) x 5000:      0.0022s
deque.popleft() x 5000:  0.0002s (should be noticeably faster)
deque(['left side', 'right side'])


### Item 104 — Know How to Use `heapq` for Priority Queues

When you repeatedly need "the smallest (or highest-priority) item so far" out of a growing collection, re-sorting a list every time is wasteful. `heapq` maintains a **min-heap** on top of a plain list: `heapq.heappush(heap, item)` adds in O(log n), `heapq.heappop(heap)` removes and returns the smallest item, also in O(log n) — much cheaper than repeatedly sorting.


In [5]:
import heapq

# A priority queue of (priority, task) tuples -- heapq compares tuples element by element
tasks = []
heapq.heappush(tasks, (3, 'write report'))
heapq.heappush(tasks, (1, 'fix critical bug'))
heapq.heappush(tasks, (2, 'review pull request'))

while tasks:
    priority, task = heapq.heappop(tasks)   # always returns the LOWEST priority number first
    print(f'priority {priority}: {task}')

# nlargest/nsmallest: convenient helpers built on the same heap machinery
numbers = [5, 1, 9, 3, 7, 2, 8]
print('3 largest:', heapq.nlargest(3, numbers))
print('3 smallest:', heapq.nsmallest(3, numbers))


priority 1: fix critical bug
priority 2: review pull request
priority 3: write report
3 largest: [9, 8, 7]
3 smallest: [1, 2, 3]


### Item 105 — Use `datetime` Instead of `time` for Local Clocks

The older `time` module's timezone handling is a thin, error-prone wrapper around C library functions tied to the *local system's* timezone setting — genuinely difficult to use correctly across multiple timezones. `datetime` (with `zoneinfo` for actual timezone databases, Python 3.9+) represents times explicitly as timezone-aware objects, making conversions between timezones (and comparisons across them) correct and explicit instead of implicit and fragile.


In [6]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

utc_now = datetime.now(timezone.utc)
print('UTC:', utc_now)

# Explicit, correct timezone conversion -- no ambiguity about which timezone applies
ny_time = utc_now.astimezone(ZoneInfo('America/New_York'))
tokyo_time = utc_now.astimezone(ZoneInfo('Asia/Tokyo'))
print('New York:', ny_time)
print('Tokyo:', tokyo_time)

# Comparing timezone-AWARE datetimes is always correct, regardless of which zones they're in
print('Same instant in time:', utc_now == ny_time == tokyo_time)


UTC: 2026-07-21 16:05:03.143060+00:00
New York: 2026-07-21 12:05:03.143060-04:00
Tokyo: 2026-07-22 01:05:03.143060+09:00
Same instant in time: True


### Item 106 — Use `decimal` When Precision Is Paramount

`float` uses binary floating-point, which cannot represent most decimal fractions exactly (`0.1 + 0.2 != 0.3`) — fine for scientific computing, but a real bug source for money and other values requiring exact decimal precision. `decimal.Decimal` represents numbers in exact base-10 arithmetic instead, avoiding these rounding surprises entirely — always prefer it for financial calculations.


In [7]:
print(0.1 + 0.2)                 # 0.30000000000000004 -- the classic float trap
print(0.1 + 0.2 == 0.3)           # False!

from decimal import Decimal

price = Decimal('19.99')
tax_rate = Decimal('0.08')
total = price * (1 + tax_rate)
print(total)                       # exact decimal arithmetic, no binary rounding error

print(Decimal('0.1') + Decimal('0.2') == Decimal('0.3'))   # True -- exact as expected

# ALWAYS construct Decimal from a STRING, not a float -- otherwise you inherit the float's imprecision
print(Decimal(0.1))          # constructing from a float bakes in the float's imprecision
print(Decimal('0.1'))         # constructing from a string is exact


0.30000000000000004
False
21.5892
True
0.1000000000000000055511151231257827021181583404541015625
0.1


### Item 107 — Make `pickle` Serialization Maintainable with `copyreg`

`pickle` serializes Python objects by default based on their class's current attributes — but if you later add a new field with a default value, `pickle.loads()` on *old* pickled data (created before the field existed) will produce an object missing that attribute, silently breaking code that expects it. `copyreg.pickle(cls, reducer_function)` lets you control serialization/deserialization explicitly, so you can supply defaults for missing fields and keep old pickled data compatible as your classes evolve.


In [8]:
import pickle
import copyreg

class GameState:
    def __init__(self, level=1, lives=3, points=0):
        self.level = level
        self.lives = lives
        self.points = points   # imagine this field was added AFTER some data was already pickled

def pickle_game_state(state):
    kwargs = state.__dict__
    return unpickle_game_state, (kwargs,)

def unpickle_game_state(kwargs):
    return GameState(**kwargs)   # always goes through __init__, so NEW defaults apply to OLD data

copyreg.pickle(GameState, pickle_game_state)

# Simulate 'old' data that was pickled BEFORE the 'points' field existed
old_state = GameState(level=2, lives=1)
del old_state.__dict__['points']   # simulate the field genuinely not existing in old data
serialized = pickle.dumps(old_state)

restored = pickle.loads(serialized)
print(restored.level, restored.lives, restored.points)   # 'points' backfilled via __init__'s default


2 1 0
